# FlowNetS

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FlowNetSimple(nn.Module):
    """
    FlowNetSimple (FlowNetS) model for optical flow estimation.
    """
    def __init__(self):
        super(FlowNetSimple, self).__init__()

        # encoder layers

        # input channels = 6 (stacking two RGB images)
        self.conv1 = nn.Conv2d(6, 64, kernel_size=7, stride=2, padding=3)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=5, stride=2, padding=2)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=5, stride=2, padding=2)
        self.conv3_1 = nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1)
        self.conv4 = nn.Conv2d(256, 512, kernel_size=3, stride=2, padding=1)
        self.conv4_1 = nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1)
        self.conv5 = nn.Conv2d(512, 512, kernel_size=3, stride=2, padding=1)
        self.conv5_1 = nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1)
        self.conv6 = nn.Conv2d(512, 1024, kernel_size=3, stride=2, padding=1)
        self.conv6_1 = nn.Conv2d(1024, 1024, kernel_size=3, stride=1, padding=1)

        # decoder layers
        self.deconv5 = nn.ConvTranspose2d(1024, 512, kernel_size=4, stride=2, padding=1)
        self.predict_flow6 = nn.Conv2d(1024, 2, kernel_size=3, stride=1, padding=1)

        # Input: 1026 channels.
        # After upsampling, you concatenate:
        # The upsampled features from the previous deconv (512 channels from deconv5)
        # The skip connection from the encoder (conv5_1, 512 channels)
        # The upsampled flow prediction from the previous scale (2 channels)
        # 512 (deconv5) + 512 (conv5_1) + 2 (upsampled flow5) = 1026 channels

        # e.g. 
        # upsampled layer (low res view) 
        # + skip connection (earlier high res view) 
        # + upsampled previous optical flow pred. for refinement
        self.deconv4 = nn.ConvTranspose2d(1026, 256, kernel_size=4, stride=2, padding=1)
        self.predict_flow5 = nn.Conv2d(1026, 2, kernel_size=3, stride=1, padding=1)
        
        self.deconv3 = nn.ConvTranspose2d(770, 128, kernel_size=4, stride=2, padding=1)
        self.predict_flow4 = nn.Conv2d(770, 2, kernel_size=3, stride=1, padding=1)
        
        self.deconv2 = nn.ConvTranspose2d(386, 64, kernel_size=4, stride=2, padding=1)
        self.predict_flow3 = nn.Conv2d(386, 2, kernel_size=3, stride=1, padding=1)

        self.predict_flow2 = nn.Conv2d(194, 2, kernel_size=3, stride=1, padding=1)

        
    
    def forward(self, x):
        # Encoder: extract features at multiple scales
        out_conv1 = F.relu(self.conv1(x))         # [B, 64, H/2, W/2]
        out_conv2 = F.relu(self.conv2(out_conv1)) # [B, 128, H/4, W/4]
        out_conv3 = F.relu(self.conv3(out_conv2)) # [B, 256, H/8, W/8]
        out_conv3_1 = F.relu(self.conv3_1(out_conv3))
        out_conv4 = F.relu(self.conv4(out_conv3_1))   # [B, 512, H/16, W/16]
        out_conv4_1 = F.relu(self.conv4_1(out_conv4))
        out_conv5 = F.relu(self.conv5(out_conv4_1))   # [B, 512, H/32, W/32]
        out_conv5_1 = F.relu(self.conv5_1(out_conv5))
        out_conv6 = F.relu(self.conv6(out_conv5_1))   # [B, 1024, H/64, W/64]
        out_conv6_1 = F.relu(self.conv6_1(out_conv6))

        # early high res layers capture smaller, fine movement
        # later low res layers capture large movement
        # used for skip connections in decoder (like U-net)

        # Decoder
        # upsampled, concatenate, predict flow, move on to next higher resolution
        # Deconvolution (coarse) + Skip connection (fine) + prev. flow prediction (refinement)
        

        flow6 = self.predict_flow6(out_conv6_1)
        
        # Upsample flow by 2 (each conv layer divided image dims by 2)
        flow6_up = F.interpolate(flow6, scale_factor=2, mode='bilinear', align_corners=False)

        # Deconvolution conv6_1 to conv5
        deconv5 = F.relu(self.deconv5(out_conv6_1))
        
        # Concat and predict next upsampled flow
        concat5 = torch.cat([deconv5, out_conv5_1, flow6_up], dim=1) # should all match now
        flow5 = self.predict_flow5(concat5)
        
        # upsample and continue
        flow5_up = F.interpolate(flow5, scale_factor=2, mode='bilinear', align_corners=False)
        deconv4 = F.relu(self.deconv4(concat5))

        concat4 = torch.cat([deconv4, out_conv4_1, flow5_up], dim=1)
        flow4 = self.predict_flow4(concat4)
        flow4_up = F.interpolate(flow4, scale_factor=2, mode='bilinear', align_corners=False)
        deconv3 = F.relu(self.deconv3(concat4))

        concat3 = torch.cat([deconv3, out_conv3_1, flow4_up], dim=1)
        flow3 = self.predict_flow3(concat3)
        flow3_up = F.interpolate(flow3, scale_factor=2, mode='bilinear', align_corners=False)
        deconv2 = F.relu(self.deconv2(concat3))

        concat2 = torch.cat([deconv2, out_conv2, flow3_up], dim=1)
        flow2 = self.predict_flow2(concat2)

        return flow2

# FlowNetCorr

- process both images seperately and then correlates them to generate flow fields

- Correlation layer computes how similar each pixel in image 1 is to every pixel in image 2 is (within a search window). This outputs the *correlation volume*.

Parts:

1. Feature Extraction - extract features independently
2. Correlation Layer - compute pixel similarity between both images combined
3. Decoder/Refinement - upsample to flow fields

### Correlation Layer

- max_displacement: hyperparameter for the search window (how many pixels to look at for correlation)
- Correlation search window formula: (2*max_displacement+1)²

In [2]:
class CorrelationLayer(nn.Module):
    def __init__(self, max_displacement=20):
        """
        Correlation layer for FlowNetC
        
        Args:
            max_displacement: maximum pixel displacement to consider
        """
        super(CorrelationLayer, self).__init__()
        self.max_displacement = max_displacement

    def forward(self, x1, x2):
        """
        Compute correlation between two feature maps
        
        Args:
            x1: first feature map (B, C, H, W)
            x2: second feature map (B, C, H, W)
            
        Returns:
            correlation volume (B, (2*max_displacement+1)^2, H, W)
        """

        # Get dimensions
        batch_size, channels, height, width = x1.size()

        # Pad x2 by max_displacement to handle boundary effects
        # replicate = copy edge values
        pad_size = self.max_displacement
        x2_padded = F.pad(x2, (pad_size, pad_size, pad_size, pad_size), mode='replicate')

        # Init correlation volume
        correlation_size = (2 * self.max_displacement + 1) ** 2
        correlation = torch.zeros(batch_size, correlation_size, height, width, device=x1.device)

        # For every pixel (i, j), iterate over search window 
        # (comparing every pixels in x1 to a window of pixels in x2)
        
        for i in range(height):
            for j in range(width):
                # Extract window from x2_padded
                window = x2_padded[:, :, 
                                 i:i + 2 * self.max_displacement + 1, 
                                 j:j + 2 * self.max_displacement + 1]
                
                # Compute correlation with pixel (i,j) from x1
                pixel_features = x1[:, :, i:i+1, j:j+1] # (B, C, 1, 1)

                # Reshape for broadcasting
                pixel_features = pixel_features.view(batch_size, channels, 1, 1)

                # Compute dot product (correlation)
                corr = torch.sum(pixel_features * window, dim=1) # (B, H_window, W_window)
                # sums along channel dimension

                # Flatten the correlation window
                corr_flat = corr.view(batch_size, -1) # (B, (2*max_displacement+1)^2)

                # Store in correlation volume
                correlation[:, :, i, j] = corr_flat 
                # each pixel has correlation dot product values with each pixel in window
                # stored along sort of the channel dim

        return correlation

In [42]:
class FlowNetCorrelation(nn.Module):

    def __init__(self, max_displacement=10, redir_dim=32):
        super(FlowNetCorrelation, self).__init__()

        # ! d = 20 in paper, but it should be d = 10 for the equation to satisfy and result in 441 correlation channels

        self.correlation_dim = (max_displacement * 2 + 1)**2
        self.conv_redir = redir_dim
        
        # 1. Feature extractors (same architecture for both images)
        # but seperately
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=5, stride=2, padding=2)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=5, stride=2, padding=2)

        # self.correlation_dim + self.conv_redir = 441 + 32 = 473 (in original paper)
        self.conv3_1 = nn.Conv2d(self.correlation_dim + self.conv_redir, 256, kernel_size=3, stride=1, padding=1)
        self.conv4 = nn.Conv2d(256, 512, kernel_size=3, stride=2, padding=1) # add 32 from conv_redir
        self.conv4_1 = nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1)
        self.conv5 = nn.Conv2d(512, 512, kernel_size=3, stride=2, padding=1)
        self.conv5_1 = nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1)
        self.conv6 = nn.Conv2d(512, 1024, kernel_size=3, stride=2, padding=1)
        self.conv6_1 = nn.Conv2d(1024, 1024, kernel_size=3, stride=1, padding=1)

        # Correlation layer / cost volume
        self.corr = CorrelationLayer(max_displacement=max_displacement)
        self.conv_redir = nn.Conv2d(256, 32, kernel_size=1, stride=1) # applied conv on image 1 that is concatenated to correlation layer
        self.corr_conv = nn.Conv2d(1681, 256, kernel_size=3, stride=1, padding=1)

        # Decoder layers
        self.deconv5 = nn.ConvTranspose2d(1024, 512, kernel_size=4, stride=2, padding=1)
        self.predict_flow6 = nn.Conv2d(1024, 2, kernel_size=3, stride=1, padding=1)

        # Input: 1026 channels
        # After upsampling, you concatenate:
        # - The upsampled features from the previous deconv (512 channels from deconv5)
        # - The skip connection from the encoder (conv5_1, 512 channels)
        # - The upsampled flow prediction from the previous scale (2 channels)
        # 512 + 512 + 2 = 1026 channels
        self.deconv4 = nn.ConvTranspose2d(1026, 256, kernel_size=4, stride=2, padding=1)
        self.predict_flow5 = nn.Conv2d(1026, 2, kernel_size=3, stride=1, padding=1)

        self.deconv3 = nn.ConvTranspose2d(770, 128, kernel_size=4, stride=2, padding=1)
        self.predict_flow4 = nn.Conv2d(770, 2, kernel_size=3, stride=1, padding=1)

        self.deconv2 = nn.ConvTranspose2d(386, 64, kernel_size=4, stride=2, padding=1)
        self.predict_flow3 = nn.Conv2d(386, 2, kernel_size=3, stride=1, padding=1)

        self.predict_flow2 = nn.Conv2d(194, 2, kernel_size=3, stride=1, padding=1)

    def forward(self, x1, x2):
        # Extract features from both images (same architecture)
        out_conv1_1 = F.relu(self.conv1(x1))
        out_conv2_1 = F.relu(self.conv2(out_conv1_1))
        out_conv3_1 = F.relu(F.relu(self.conv3(out_conv2_1)))
        
        # Same for second image
        out_conv1_2 = F.relu(self.conv1(x2))
        out_conv2_2 = F.relu(self.conv2(out_conv1_2))
        out_conv3_2 = F.relu(F.relu(self.conv3(out_conv2_2)))

        # apply at the conv3 level
        correlation = self.corr(out_conv3_1, out_conv3_2)  # (B, 1681, H, W) if max_disp=20
        # (20 * 2 + 1)^2 = 1681 (correlation dim)

        # concat correlation_layer w/ conv_redir
        redir = F.relu(self.conv_redir(out_conv3_1))  # (B, redir_dim, H, W)
        concat = torch.cat([correlation, redir], dim=1) # (B, corr_dim + redir_dim, H, W)
        in_conv3_1 = self.conv3_1(concat) # (B, 256, H, W)

        # Decoder (upsampling and flow prediction at multiple scales)
        # The following is a typical decoder structure, similar to FlowNetS:
        out_conv4 = F.relu(self.conv4(in_conv3_1))
        out_conv4_1 = F.relu(self.conv4_1(out_conv4))
        out_conv5 = F.relu(self.conv5(out_conv4_1))
        out_conv5_1 = F.relu(self.conv5_1(out_conv5))
        out_conv6 = F.relu(self.conv6(out_conv5_1))
        out_conv6_1 = F.relu(self.conv6_1(out_conv6))

        # Predict flow at the coarsest level
        flow6 = self.predict_flow6(out_conv6_1)
        flow6_up = F.interpolate(flow6, scale_factor=2, mode='bilinear', align_corners=False)
        deconv5 = F.relu(self.deconv5(out_conv6_1))
        concat5 = torch.cat([deconv5, out_conv5_1, flow6_up], dim=1)
        flow5 = self.predict_flow5(concat5)
        flow5_up = F.interpolate(flow5, scale_factor=2, mode='bilinear', align_corners=False)
        deconv4 = F.relu(self.deconv4(concat5))
        concat4 = torch.cat([deconv4, out_conv4_1, flow5_up], dim=1)
        flow4 = self.predict_flow4(concat4)
        flow4_up = F.interpolate(flow4, scale_factor=2, mode='bilinear', align_corners=False)
        deconv3 = F.relu(self.deconv3(concat4))
        concat3 = torch.cat([deconv3, out_conv3_1, flow4_up], dim=1)
        flow3 = self.predict_flow3(concat3)
        flow3_up = F.interpolate(flow3, scale_factor=2, mode='bilinear', align_corners=False)
        deconv2 = F.relu(self.deconv2(concat3))
        concat2 = torch.cat([deconv2, out_conv2_1, flow3_up], dim=1)
        flow2 = self.predict_flow2(concat2)

        # Return the finest flow prediction
        return flow2


        
def test_flownet_corr():
    model = FlowNetCorrelation()
    x1 = torch.randn(1, 3, 128, 128)
    x2 = torch.randn(1, 3, 128, 128)
    
    try:
        output = model(x1, x2)
        print("Success! Output shape:", output.shape)
    except Exception as e:
        print("Error:", e)
        print("This is expected - we haven't implemented the decoder yet!")

test_flownet_corr()

Success! Output shape: torch.Size([1, 2, 32, 32])


### Dummy test

In [43]:
# Instantiate the model
model = FlowNetSimple()

# Create a dummy input: batch size 1, 6 channels, 128x128 image
x = torch.randn(1, 6, 128, 128)

# Forward pass
flow = model(x)

# Print the output shape
print(flow.shape)  # Should be [1, 2, H, W] (H and W will be smaller than 128, depending on architecture)

torch.Size([1, 2, 32, 32])


### Training

In [44]:
# Loss function
# euclidean distance between predicted and gt flow for each pixel

def endpoint_error(pred_flow, gt_flow):
    """
    pred_flow: (B, 2, H, W)
    gt_flow: (B, 2, H, W)
    Returns: scalar loss (mean EPE over all pixels and batch)
    """
    # pred_flow: (B, 2, H_pred, W_pred)
    # flow_gt:   (B, 2, H_gt, W_gt)
    if pred_flow.shape[-2:] != flow_gt.shape[-2:]:
        pred_flow = F.interpolate(pred_flow, size=flow_gt.shape[-2:], mode='bilinear', align_corners=False)
    return torch.norm(pred_flow - gt_flow, dim=1).mean()

In [45]:
import numpy as np

def read_flo_file(filename):
    """
    Read .flo file in Sintel format.
    Returns: numpy array of shape (H, W, 2)
    """
    with open(filename, 'rb') as f:
        magic = np.frombuffer(f.read(4), np.float32)[0]
        if magic != 202021.25:
            raise Exception('Magic number incorrect. Invalid .flo file')
        w = np.frombuffer(f.read(4), np.int32)[0]
        h = np.frombuffer(f.read(4), np.int32)[0]
        data = np.frombuffer(f.read(), np.float32)
        data = np.reshape(data, (h, w, 2))
        return data

In [46]:
import torchvision.transforms as T

img_transform = T.Compose([
    # T.Resize((128, 128)),  # Resize to 128x128 for debugging
    T.Resize((384, 512)),
    T.ToTensor(),          # Converts PIL Image to tensor and scales to [0,1]
    T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])  # Optional: normalize to [-1, 1]
])

In [47]:
import os
from torch.utils.data import Dataset
from PIL import Image
import torch

class SintelDataset(Dataset):
    def __init__(self, root_dir, split='training', render_pass='clean', transform=None):
        """
        root_dir: path to Sintel dataset (e.g., '/pub/tyleryy/Models-from-Scratch/data/Sintel')
        split: 'training' or 'test'
        render_pass: 'clean' or 'final'
        transform: optional transform to apply to images/flow
        """
        self.img_dir = os.path.join(root_dir, split, render_pass)
        self.flow_dir = os.path.join(root_dir, split, 'flow')
        self.transform = transform

        self.samples = []
        # For each scene, get all consecutive frame pairs
        for scene in sorted(os.listdir(self.img_dir)):
            img_scene_dir = os.path.join(self.img_dir, scene)
            flow_scene_dir = os.path.join(self.flow_dir, scene)
            frames = sorted([f for f in os.listdir(img_scene_dir) if f.endswith('.png')])
            for i in range(len(frames) - 1):
                img1 = os.path.join(img_scene_dir, frames[i])
                img2 = os.path.join(img_scene_dir, frames[i+1])
                flow = os.path.join(flow_scene_dir, frames[i].replace('.png', '.flo'))
                self.samples.append((img1, img2, flow))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img1_path, img2_path, flow_path = self.samples[idx]
        img1 = Image.open(img1_path).convert('RGB')
        img2 = Image.open(img2_path).convert('RGB')
        flow = read_flo_file(flow_path)  # (H, W, 2)

        # Convert to tensors and permute to (C, H, W)
        # img1 = torch.from_numpy(np.array(img1)).permute(2, 0, 1).float() / 255.0
        # img2 = torch.from_numpy(np.array(img2)).permute(2, 0, 1).float() / 255.0
        flow = torch.from_numpy(flow).permute(2, 0, 1).float()  # (2, H, W)

        if self.transform:
            img1 = self.transform(img1)
            img2 = self.transform(img2)
        else:
            img1 = T.ToTensor()(img1)
            img2 = T.ToTensor()(img2)


        # print(img1.shape, img2.shape, flow.shape)

        return img1, img2, flow

In [48]:
from torch.utils.data import DataLoader

dataset = SintelDataset('/pub/tyleryy/Models-from-Scratch/data/Sintel', render_pass='clean', transform=img_transform)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True, num_workers=2)

In [49]:
for img1, img2, flow in dataloader:
    print("img1 shape:", img1.shape)  # (B, 3, H, W)
    print("img2 shape:", img2.shape)  # (B, 3, H, W)
    print("flow shape:", flow.shape)  # (B, 2, H, W)
    break

/tmp/tyleryy/40360754/ipykernel_667098/2789577862.py:42: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:203.)
  flow = torch.from_numpy(flow).permute(2, 0, 1).float()  # (2, H, W)
/tmp/tyleryy/40360754/ipykernel_667098/2789577862.py:42: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/uti

img1 shape: torch.Size([8, 3, 384, 512])
img2 shape: torch.Size([8, 3, 384, 512])
flow shape: torch.Size([8, 2, 436, 1024])


In [50]:
# TODO: do data augmentation/transformation

In [ ]:
import torch.optim as optim

model = FlowNetCorrelation()  # or whatever your class is called
model = model.cuda()  # if using GPU

optimizer = optim.Adam(model.parameters(), lr=1e-4)

num_epochs = 100  # or however many you want

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for i, (img1, img2, flow_gt) in enumerate(dataloader):
        img1 = img1.cuda()
        img2 = img2.cuda()
        flow_gt = flow_gt.cuda()

        optimizer.zero_grad()
        pred_flow = model(img1, img2)
        loss = endpoint_error(pred_flow, flow_gt)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if (i + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(dataloader)}], Loss: {loss.item():.4f}")
            torch.save(model.state_dict(), f"flownet_epoch{epoch+1}.pth")

    avg_loss = running_loss / len(dataloader)
    print(f"Epoch [{epoch+1}/{num_epochs}] Average Loss: {avg_loss:.4f}")

torch.save(model.state_dict(), f"flownet_epoch{epoch+1}.pth")

/tmp/tyleryy/40360754/ipykernel_667098/2789577862.py:42: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:203.)
  flow = torch.from_numpy(flow).permute(2, 0, 1).float()  # (2, H, W)
/tmp/tyleryy/40360754/ipykernel_667098/2789577862.py:42: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/uti

Epoch [1/100], Step [10/131], Loss: 8.1736
Epoch [1/100], Step [20/131], Loss: 10.0410
Epoch [1/100], Step [30/131], Loss: 14.7425
Epoch [1/100], Step [40/131], Loss: 18.4135
Epoch [1/100], Step [50/131], Loss: 16.0102
Epoch [1/100], Step [60/131], Loss: 21.0346
Epoch [1/100], Step [70/131], Loss: 9.9169
Epoch [1/100], Step [80/131], Loss: 12.8851
Epoch [1/100], Step [90/131], Loss: 17.8740
Epoch [1/100], Step [100/131], Loss: 12.1367
Epoch [1/100], Step [110/131], Loss: 7.1693
Epoch [1/100], Step [120/131], Loss: 12.2748
Epoch [1/100], Step [130/131], Loss: 25.7282
Epoch [1/100] Average Loss: 13.6589


/tmp/tyleryy/40360754/ipykernel_667098/2789577862.py:42: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:203.)
  flow = torch.from_numpy(flow).permute(2, 0, 1).float()  # (2, H, W)
/tmp/tyleryy/40360754/ipykernel_667098/2789577862.py:42: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/uti

Epoch [2/100], Step [10/131], Loss: 16.2437
Epoch [2/100], Step [20/131], Loss: 8.4291
Epoch [2/100], Step [30/131], Loss: 14.1781
Epoch [2/100], Step [40/131], Loss: 8.6224
Epoch [2/100], Step [50/131], Loss: 3.6138
Epoch [2/100], Step [60/131], Loss: 16.6709
Epoch [2/100], Step [70/131], Loss: 6.5531
Epoch [2/100], Step [80/131], Loss: 12.2533
Epoch [2/100], Step [90/131], Loss: 12.5572
Epoch [2/100], Step [100/131], Loss: 24.6183
Epoch [2/100], Step [110/131], Loss: 4.1623
Epoch [2/100], Step [120/131], Loss: 38.3069
Epoch [2/100], Step [130/131], Loss: 5.7653
Epoch [2/100] Average Loss: 13.3453


/tmp/tyleryy/40360754/ipykernel_667098/2789577862.py:42: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:203.)
  flow = torch.from_numpy(flow).permute(2, 0, 1).float()  # (2, H, W)
/tmp/tyleryy/40360754/ipykernel_667098/2789577862.py:42: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/uti

...

Epoch [26/100], Step [130/131], Loss: 3.2568
Epoch [26/100] Average Loss: 9.2269


/tmp/tyleryy/40360754/ipykernel_667098/2789577862.py:42: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:203.)
  flow = torch.from_numpy(flow).permute(2, 0, 1).float()  # (2, H, W)
/tmp/tyleryy/40360754/ipykernel_667098/2789577862.py:42: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/uti

Epoch [27/100], Step [10/131], Loss: 10.8400
Epoch [27/100], Step [20/131], Loss: 4.1720
Epoch [27/100], Step [30/131], Loss: 3.7504
Epoch [27/100], Step [40/131], Loss: 5.8799
Epoch [27/100], Step [50/131], Loss: 6.0284
Epoch [27/100], Step [60/131], Loss: 9.8024
Epoch [27/100], Step [70/131], Loss: 9.6656
Epoch [27/100], Step [80/131], Loss: 13.6273
Epoch [27/100], Step [90/131], Loss: 13.6396
Epoch [27/100], Step [100/131], Loss: 4.4461
Epoch [27/100], Step [110/131], Loss: 5.1866
Epoch [27/100], Step [120/131], Loss: 13.9585
Epoch [27/100], Step [130/131], Loss: 14.2635
Epoch [27/100] Average Loss: 8.8970


/tmp/tyleryy/40360754/ipykernel_667098/2789577862.py:42: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:203.)
  flow = torch.from_numpy(flow).permute(2, 0, 1).float()  # (2, H, W)
/tmp/tyleryy/40360754/ipykernel_667098/2789577862.py:42: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/uti

Epoch [28/100], Step [10/131], Loss: 4.0988
Epoch [28/100], Step [20/131], Loss: 8.7858
Epoch [28/100], Step [30/131], Loss: 12.5216
Epoch [28/100], Step [40/131], Loss: 6.7023
Epoch [28/100], Step [50/131], Loss: 10.1328
Epoch [28/100], Step [60/131], Loss: 2.5457
Epoch [28/100], Step [70/131], Loss: 8.7719
Epoch [28/100], Step [80/131], Loss: 11.0699
Epoch [28/100], Step [90/131], Loss: 15.0691
